In [20]:
import os
import re
import json

def clean_compliance_text(raw_text):
    """Sanitizes messy federal text files for analytical use."""
    
    # 1. Fix broken hyphenated words (e.g., "equip- \n ment" -> "equipment")
    text = re.sub(r'-\s*\n\s*', '', raw_text)
    
    # 2. Remove isolated page numbers (e.g., a number sitting alone on a line)
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)
    
    # 3. Remove hard line breaks within sentences, but keep paragraph breaks
    # This replaces single newlines with a space, but leaves double newlines alone
    text = re.sub(r'(?<!\n)\n(?!\n)', ' ', text)
    
    # 4. Normalize whitespace (remove double spaces, trim edges)
    text = re.sub(r'[ \t]+', ' ', text)
    text = text.strip()
    
    return text

def chunk_text(clean_text, chunk_size=1000):
    """Splits the clean text into logical chunks based on paragraphs."""
    paragraphs = clean_text.split('\n\n')
    valid_chunks = [p.strip() for p in paragraphs if len(p.strip()) > 50]
    return valid_chunks

def process_directory(input_dir, output_dir):
    """The main ETL Pipeline: Extracts, Transforms, and Loads the data."""
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    for filename in os.listdir(input_dir):
        if filename.endswith(".txt"):
            input_path = os.path.join(input_dir, filename)
            
            # EXTRACT
            print(f"Extracting: {filename}...")
            with open(input_path, 'r', encoding='utf-8', errors='ignore') as file:
                raw_data = file.read()
            
            # TRANSFORM
            cleaned_data = clean_compliance_text(raw_data)
            structured_chunks = chunk_text(cleaned_data)
            
            # LOAD
            output_data = {
                "source_file": filename,
                "total_chunks": len(structured_chunks),
                "data": structured_chunks
            }
            
            output_filename = filename.replace('.txt', '_clean.json')
            output_path = os.path.join(output_dir, output_filename)
            
            with open(output_path, 'w', encoding='utf-8') as json_file:
                json.dump(output_data, json_file, indent=4)
                
            print(f"Successfully loaded clean data to: {output_filename}")

# Run the ETL Pipeline
if __name__ == "__main__":
    process_directory("C:/Users/MAC/rag_detector", ".rag_detector/")

Extracting: env.txt...
Successfully loaded clean data to: env_clean.json
Extracting: title14.txt...
Successfully loaded clean data to: title14_clean.json
Extracting: title14_cfr.txt...
Successfully loaded clean data to: title14_cfr_clean.json


In [21]:
import json

with open(".rag_detector/title14_clean.json", "r", encoding="utf-8") as f:
    data = json.load(f)

type(data), len(data)

(dict, 3)

In [22]:
#Use pandas to load cleanly
import pandas as pd

df = pd.DataFrame(data)
df.head()

,source_file,total_chunks,data
0,title14.txt,4991,[Title 14 CFR ] [Code of Federal Regulations (...
1,title14.txt,4991,Containing a codification of documents of gene...
2,title14.txt,4991,"As of January 1, 2025 Published by the Office ..."
3,title14.txt,4991,Legal Status and Use of Seals and Logos The se...
4,title14.txt,4991,It is prohibited to use NARA's official seal a...


In [23]:
#JSOn did not stoe a list of dicts
#Instead stored row-wise records
#data is a text chunk
#source_file and total_chunks are metadata

df.loc[0, "data"][:1000]

'[Title 14 CFR ] [Code of Federal Regulations (annual edition) - January 1, 2025 Edition] [From the U.S. Government Publishing Office]'

In [24]:
#Important for embeddings: checking chunk length distribution
df["data"].str.len().describe()

count    4991.000000
mean      668.320377
std       903.466054
min        51.000000
25%       107.000000
50%       279.000000
75%       833.500000
max      5727.000000
Name: data, dtype: float64

In [25]:
#To check chunk consistency
df["total_chunks"].unique()

array([4991])

In [26]:
#Sanity check to ensure RAG readiness and data is adapted to structure
assert len(df) > 0
assert df["data"].str.strip().ne("").all()
assert df["source_file"].notna().all()

In [27]:
#Repeat with other text file
df_cfr = pd.read_json(".rag_detector/title14_cfr_clean.json")
df_cfr.head()

,source_file,total_chunks,data
0,title14_cfr.txt,13,Code of Federal Regulations Title 14 - Aeronau...
1,title14_cfr.txt,13,"61Certification: Pilots, flight instructors, a..."
2,title14_cfr.txt,13,63Certification: Flight crewmembers other than...
3,title14_cfr.txt,13,65Certification: Airmen other than flight crew...
4,title14_cfr.txt,13,68Requirements for operating certain small air...


In [28]:
len(df), len(df_cfr)

(4991, 13)

In [29]:
#A look at JSON column names
df.columns.tolist()

['source_file', 'total_chunks', 'data']

In [30]:
#Adding more metadata columns to JSON
df["chunk_id"] = df.index
df["char_len"] = df["data"].str.len()

In [31]:
df.columns.tolist()

['source_file', 'total_chunks', 'data', 'chunk_id', 'char_len']

In [32]:
assert df.shape[1] == 5
assert df["data"].str.strip().ne("").all()
assert df["total_chunks"].nunique() == 1

In [33]:
#Handoff for embeddings
texts = df["data"].tolist()

In [34]:
#To generate realistic operations records

import pandas as pd
import random
from datetime import datetime, timedelta

def generate_yard_log(num_records=50):
    # Setup baseline data
    equipment_types = ['Gantry Crane', 'Reach Stacker', 'Forklift', 'Terminal Tractor']
    locations = ['Bay 1', 'Bay 2', 'Bay 4', 'Maintenance Track', 'Main Pedestrian Crosswalk', 'High-Voltage Line B']
    incident_types = ['None', 'None', 'None', 'Hydraulic Leak', 'Load Imbalance', 'Proximity Warning']
    
    data = []
    base_time = datetime(2026, 3, 29, 6, 0) # Starting at 06:00 AM
    
    for i in range(num_records):
        equip = random.choice(equipment_types)
        loc = random.choice(locations)
        
        # Injecting a few intentional violations for the AI to catch
        if i == 12: 
            shift_hours = 14.5 # Violation: Over allowable shift hours
            incident = 'None'
        elif i == 27:
            loc = 'Main Pedestrian Crosswalk'
            incident = 'Hydraulic Leak' # Violation: Hazardous spill near foot traffic
            shift_hours = 8.0
        elif i == 42:
            loc = 'High-Voltage Line B'
            incident = 'Proximity Warning' # Violation: Clearance limits
            shift_hours = 10.0
        else:
            shift_hours = round(random.uniform(6.0, 11.5), 1)
            incident = random.choice(incident_types)
            
        record_time = base_time + timedelta(minutes=random.randint(5, 30) * i)
        
        data.append({
            'Log_ID': f"LOG-{1000 + i}",
            'Timestamp': record_time.strftime('%Y-%m-%d %H:%M'),
            'Equipment_ID': f"{equip[:3].upper()}-{random.randint(10, 99)}",
            'Equipment_Type': equip,
            'Location': loc,
            'Operator_Shift_Hours': shift_hours,
            'Payload_Weight_lbs': random.randint(10000, 65000),
            'Reported_Incident': incident
        })
        
    df = pd.DataFrame(data)
    df.to_csv('daily_yard_log.csv', index=False)
    print("Successfully generated daily_yard_log.csv with 50 records.")
    return df

# Run the generator
yard_log_df = generate_yard_log()
yard_log_df.head()

Successfully generated daily_yard_log.csv with 50 records.


,Log_ID,Timestamp,Equipment_ID,Equipment_Type,Location,Operator_Shift_Hours,Payload_Weight_lbs,Reported_Incident
0,LOG-1000,2026-03-29 06:00,FOR-28,Forklift,Maintenance Track,6.0,36118,Load Imbalance
1,LOG-1001,2026-03-29 06:28,TER-75,Terminal Tractor,High-Voltage Line B,8.4,46275,None
2,LOG-1002,2026-03-29 06:30,REA-84,Reach Stacker,Bay 4,10.7,42150,Load Imbalance
3,LOG-1003,2026-03-29 07:15,GAN-27,Gantry Crane,High-Voltage Line B,9.3,62083,Load Imbalance
4,LOG-1004,2026-03-29 07:04,TER-48,Terminal Tractor,Bay 1,10.7,41311,Proximity Warning


In [ ]:
#Autonomous Agent Code
import pandas as pd
from langchain_community.chat_models import ChatOllama
from langchain_core.prompts import PromptTemplate



# Load the Ground Truth data you generated
df = pd.read_csv('daily_yard_log.csv')

# Initialize the LLM 

llm = ChatOllama(model="qwen3:14b", temperature=0.0)

# Define the strict System Prompt for the Agent
audit_prompt = PromptTemplate.from_template("""
You are an expert heavy logistics Safety and Compliance Auditor. 
Cross-reference this yard event with OSHA and FRA regulatory standards.

Event Details:
- Log ID: {log_id}
- Equipment: {equipment} ({equip_type})
- Location: {location}
- Operator Shift Hours: {shift_hours}
- Incident Reported: {incident}

Analyze the event for:
1. Operator fatigue violations (Shifts over 12.0 hours).
2. Hazardous material/spill violations in pedestrian zones.
3. Equipment clearance and proximity violations (High-Voltage lines).

OUTPUT FORMAT:
STATUS: [CLEAR or VIOLATION DETECTED]
REASON: [Brief explanation of the operational hazard]
RECOMMENDED ACTION: [Immediate floor manager action]
""")

# Create the LangChain processing pipeline
audit_chain = audit_prompt | llm

print("Starting Autonomous Yard Audit...\n" + "="*50)

# Run the Agent against a standard row AND our planted violations
test_rows = [0, 12, 27, 42] 

for index in test_rows:
    row = df.iloc[index]
    print(f"\n[ RUNNING AUDIT ON {row['Log_ID']} ]")
    
    # The Agent autonomously analyzes the data
    response = audit_chain.invoke({
        "log_id": row['Log_ID'],
        "equipment": row['Equipment_ID'],
        "equip_type": row['Equipment_Type'],
        "location": row['Location'],
        "shift_hours": row['Operator_Shift_Hours'],
        "incident": row['Reported_Incident']
    })
    
    print(response.content)
    print("-" * 50)


Starting Autonomous Yard Audit...

[ RUNNING AUDIT ON LOG-1000 ]
STATUS: CLEAR  
REASON: No violations detected. Operator shift hours (6.0) are below OSHA’s 12-hour limit for fatigue; no hazardous materials/spill incident was reported; and no proximity to high-voltage lines was indicated in the event details.  
RECOMMENDED ACTION: Conduct routine inspection of FOR-28’s load stability systems and ensure operator training on proper load balancing procedures.
--------------------------------------------------

[ RUNNING AUDIT ON LOG-1012 ]
STATUS: VIOLATION DETECTED  
REASON: Operator worked 14.5 hours, exceeding OSHA’s general safety guideline of 12-hour shifts for heavy machinery operations, increasing risk of fatigue-related errors.  
RECOMMENDED ACTION: Immediately reassign the operator to a rest period and review shift scheduling to ensure compliance with OSHA’s 29 CFR 1910.272 (fatigue management) and FRA’s 49 CFR 1244.107 (hours of service for equipment operators). Conduct a fatigu